# Topic Modelling Starter

This notebook sets up a reusable workflow for LDA and BERTopic on the news dataset.

In [1]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
#from bertopic import BERTopic

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer



In [2]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/willevans/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/willevans/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
DATA_PATH = r"/Users/willevans/Desktop/data.csv"
TEXT_COLUMN = "content"
SAMPLE_SIZE = 5000

df = pd.read_csv(DATA_PATH, encoding='latin1')
df = df.dropna(subset=[TEXT_COLUMN]).copy()
df = df[["article_id", "title", "category", TEXT_COLUMN]].copy()

if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

df.head()

/var/folders/m4/py2v9t557sn5pwgbcr52g5g40000gn/T/ipykernel_67028/2245209781.py:5: DtypeWarning: Columns (0: article_id, 1: source_id, 2: category, 3: full_content, 4: Unnamed: 12, 5: Unnamed: 13, 6: Unnamed: 14, 7: Unnamed: 15, 8: Unnamed: 16, 9: Unnamed: 17, 10: Unnamed: 18, 11: Unnamed: 19, 12: Unnamed: 20, 13: Unnamed: 21, 14: Unnamed: 22, 15: Unnamed: 23, 16: Unnamed: 24, 17: Unnamed: 25, 18: Unnamed: 26, 19: Unnamed: 27, 20: Unnamed: 28, 21: Unnamed: 29, 22: Unnamed: 30, 23: Unnamed: 31, 24: Unnamed: 32, 25: Unnamed: 33, 26: Unnamed: 34, 27: Unnamed: 35, 28: Unnamed: 36, 29: Unnamed: 37, 30: Unnamed: 38, 31: Unnamed: 39, 32: Unnamed: 40, 33: Unnamed: 41, 34: Unnamed: 42, 35: Unnamed: 43, 36: Unnamed: 44, 37: Unnamed: 45, 38: Unnamed: 46, 39: Unnamed: 47, 40: Unnamed: 48, 41: Unnamed: 49, 42: Unnamed: 50, 43: Unnamed: 51, 44: Unnamed: 52, 45: Unnamed: 53, 46: Unnamed: 54, 47: Unnamed: 55, 48: Unnamed: 56, 49: Unnamed: 57, 50: Unnamed: 58, 51: Unnamed: 59, 52: Unnamed: 60, 53: Unnam

,article_id,title,category,content
0,106461,ãBlenderãã¦ã¼ã¶ã¼ã®ãç¥­ããBlende...,YouTube,3DCGBlender \n 102628Blender Conferenceskeynot...
1,104326,Morning news brief,News,Some Americans and other foreign nationals hav...
2,94922,A colorful slice of city life: Winners of the ...,Photography,The winners of the Urban Photo Awards 2023 hav...
3,92170,"Hanwha Ocean eyes submarine exports to Canada,...",Philippines,"SEONGNAM, South Korea : South Korea's Hanwha O..."
4,103071,Banco Santander-Chile (NYSE:BSAC) Upgraded by ...,America,Banco Santander-Chile (NYSE:BSAC â Get Free ...


In [8]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [token for token in tokens if len(token) > 2 and token not in stop_words]
    return tokens

df["tokens"] = df[TEXT_COLUMN].apply(clean_text)
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

df[["title", "category", "clean_text"]].head()

,title,category,clean_text
0,ãBlenderãã¦ã¼ã¶ã¼ã®ãç¥­ããBlende...,YouTube,dcgblender blender conferenceskeynote ton roos...
1,Morning news brief,News,americans foreign nationals able leave gaza wh...
2,A colorful slice of city life: Winners of the ...,Photography,winners urban photo awards unveiled collection...
3,"Hanwha Ocean eyes submarine exports to Canada,...",Philippines,seongnam south korea south korea hanwha ocean ...
4,Banco Santander-Chile (NYSE:BSAC) Upgraded by ...,America,banco santander chile nyse bsac get free repor...


In [9]:
print(df.shape)
df["category"].fillna("Unknown").value_counts().head(10)

(5000, 6)


category
Cars                       249
Real estate                168
Love                       147
Games                      141
Politics                   132
Food                       129
Climate                    120
Artificial Intelligence    115
New Zealand                110
Space                      105
Name: count, dtype: int64

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.90,
    min_df=20,
    ngram_range=(1, 2)
)

doc_term_matrix = vectorizer.fit_transform(df["clean_text"])

In [11]:
for n in [5, 8, 10, 12]:
    lda_model = LatentDirichletAllocation(
        n_components=n,
        random_state=42,
        learning_method="batch"
    )
    lda_model.fit(doc_term_matrix)
    print(f"\nTop words for {n} topics:")
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(lda_model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")


Top words for 5 topics:
Topic 1: nov, announced, new, global, globe, newswire, president, globe newswire, united, nov globe
Topic 2: world, news, new, cup, world cup, south, watch, live, zealand, new zealand
Topic 3: new, getty, years, images, like, getty images, time, week, life, day
Topic 4: report, free, free report, quarter, recent, according, company, nyse, shares, second
Topic 5: year, thursday, said, million, israel, gaza, reported, amazon, people, hamas

Top words for 8 topics:
Topic 1: nov, announced, globe, newswire, globe newswire, nov globe, global, series, states, today
Topic 2: world, news, cup, world cup, new, watch, south, zealand, new zealand, live
Topic 3: years, new, time, life, youtube, long, car, amp, good, way
Topic 4: report, free, free report, recent, according, quarter, company, nyse, shares, quarter according
Topic 5: year, thursday, million, apple, new, reported, market, announced, business, bitcoin
Topic 6: new, like, york, new york, city, people, home, med

In [12]:
def print_lda_topics(model, feature_names, top_n=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-top_n - 1:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

feature_names = vectorizer.get_feature_names_out()
print_lda_topics(lda_model, feature_names)

Topic 1: season, food, start, new, friday, international, october, countries, event, announced
Topic 2: news, watch, live, sports, panama, apple, cbs, copyright, north, exclusive
Topic 3: years, life, time, games, google, youtube, day, new, week, car
Topic 4: report, free, free report, recent, quarter, according, company, shares, nyse, quarter according
Topic 5: thursday, year, reported, new, announced, people, bitcoin, data, company, quarter
Topic 6: new, november, amp, like, media, home, social, city, led, instagram
Topic 7: million, year, business, chief, space, net, big, health, financial, president
Topic 8: getty, images, israel, getty images, said, gaza, state, hamas, china, war
Topic 9: world, new, cup, world cup, zealand, new zealand, intelligence, artificial, artificial intelligence, press
Topic 10: new, year, old, said, sign, story, thursday, year old, music, news
Topic 11: nov, global, new, globe, newswire, globe newswire, nov globe, york, new york, market
Topic 12: united, 

In [13]:
lda_topic_matrix = lda_model.transform(doc_term_matrix)
df["lda_topic"] = lda_topic_matrix.argmax(axis=1)
df[["title", "category", "lda_topic"]].head(10)

,title,category,lda_topic
0,ãBlenderãã¦ã¼ã¶ã¼ã®ãç¥­ããBlende...,YouTube,8
1,Morning news brief,News,7
2,A colorful slice of city life: Winners of the ...,Photography,7
3,"Hanwha Ocean eyes submarine exports to Canada,...",Philippines,7
4,Banco Santander-Chile (NYSE:BSAC) Upgraded by ...,America,3
5,ç¾å½åäº¿æ æ¥ç»è´¹é­è´¨ç(å¾),COVID,7
6,"More foreign citizens, including about 400 Ame...",Food,7
7,Uzodimma accuses Ajaero of meddling in Imo pol...,Politics,7
8,Armenia's Pashinyan hopes peace deal with Azer...,Armenia,9
9,Tech Billionaires' Quest to Build a New City i...,Real estate,4


In [15]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

dictionary = Dictionary(df["tokens"])
corpus = [dictionary.doc2bow(tokens) for tokens in df["tokens"]]
topic_terms = []

for topic in lda_model.components_:
    top_indices = topic.argsort()[:-11:-1]
    topic_terms.append([feature_names[i] for i in top_indices])

coherence_model = CoherenceModel(
    topics=topic_terms,
    texts=df["tokens"],
    dictionary=dictionary,
    coherence="c_v"
)

print({"lda_coherence": coherence_model.get_coherence()})

{'lda_coherence': np.float64(0.42508752698380214)}


In [16]:
coherence_scores = {}
for n in [5, 8, 10, 12]:
    lda_model = LatentDirichletAllocation(
        n_components=n,
        random_state=42,
        learning_method="batch"
    )
    lda_model.fit(doc_term_matrix)
    
    topic_terms = []
    for topic in lda_model.components_:
        top_indices = topic.argsort()[:-11:-1]
        topic_terms.append([feature_names[i] for i in top_indices])
    
    coherence_model = CoherenceModel(
        topics=topic_terms,
        texts=df["tokens"],
        dictionary=dictionary,
        coherence="c_v"
    )
    coherence_scores[n] = coherence_model.get_coherence()

print("Coherence scores for different k:")
for k, score in coherence_scores.items():
    print(f"k={k}: {score:.4f}")

Coherence scores for different k:
k=5: 0.6030
k=8: 0.4735
k=10: 0.4772
k=12: 0.4251


In [9]:
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
bertopic_topics, bertopic_probs = topic_model.fit_transform(df["clean_text"].tolist())
df["bertopic_topic"] = bertopic_topics

2026-03-24 16:37:47,410 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-03-24 16:38:06,739 - BERTopic - Embedding - Completed ✓
2026-03-24 16:38:06,740 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-24 16:38:30,798 - BERTopic - Dimensionality - Completed ✓
2026-03-24 16:38:30,800 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-24 16:38:32,108 - BERTopic - Cluster - Completed ✓
2026-03-24 16:38:32,113 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-24 16:38:32,279 - BERTopic - Representation - Completed ✓


In [10]:
topic_info = topic_model.get_topic_info()
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1517,-1_chars_new_nov_globe,"[chars, new, nov, globe, newswire, said, marke...",[space part future inc international media gro...
1,0,488,0_quarter_recent_according_filing,"[quarter, recent, according, filing, inc, seco...",[ieq capital llc lowered position mobile inc n...
2,1,292,1_israel_gaza_hamas_israeli,"[israel, gaza, hamas, israeli, war, iran, pale...",[united nations china iran multitude arab nati...
3,2,160,2_getty_work_artificial_intelligence,"[getty, work, artificial, intelligence, htc, c...",[venturebeat presents unleashed exclusive exec...
4,3,140,3_pro_amazon_apple_samsung,"[pro, amazon, apple, samsung, iphone, deals, b...",[microsoft amazon back prime day big deal days...
5,4,126,4_film_movie_series_films,"[film, movie, series, films, show, episode, co...",[ranbir also spoke film tried find similarity ...
6,5,104,5_ukraine_russia_russian_ukrainian,"[ukraine, russia, russian, ukrainian, moscow, ...",[ukraine aim winter months cut russian militar...
7,6,100,6_crypto_bitcoin_cryptocurrency_reserve,"[crypto, bitcoin, cryptocurrency, reserve, fed...",[bitcoin cryptocurrencies advanced thursday li...
8,7,79,7_quarterback_nba_basketball_season,"[quarterback, nba, basketball, season, game, s...",[tempe ariz second time season arizona cardina...
9,8,72,8_cup_india_cricket_world,"[cup, india, cricket, world, australia, englan...",[international cricket council icc expected ea...


In [11]:
topic_model.get_topic(0)

[('quarter', np.float64(0.049808920140921656)),
 ('recent', np.float64(0.047806710485687076)),
 ('according', np.float64(0.04542645734241109)),
 ('filing', np.float64(0.04287653218780642)),
 ('inc', np.float64(0.039300233916086114)),
 ('second', np.float64(0.03669269546487431)),
 ('free', np.float64(0.035961713820082564)),
 ('nyse', np.float64(0.035516326826683216)),
 ('llc', np.float64(0.033853200976171706)),
 ('holdings', np.float64(0.033064534613084624))]

In [12]:
# Run in Jupyter to open the interactive visualisation.\n
topic_model.visualize_barchart(top_n_topics=10)

In [21]:
topic_model.visualize_topics()

In [23]:
topic_model_17 = BERTopic(nr_topics=17)
topics_17, probs_17 = topic_model_17.fit_transform(df["clean_text"].tolist())

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.